# Single Family Concat with Condos

This notebook utilizes building data to calculate volume for single family homes. Buildings nearest to Zillow points are utilized to create these calculations.

In [ ]:
# import necessary libraries
import pandas as pd
from shapely.geometry import box
import numpy as np
import geopandas as gpd
import os
import matplotlib.pyplot as plt
import fiona

# statistical libraries
#import sys
#!{sys.executable} -m pip install statsmodels
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression

In [3]:
# set option to see all data frame columns
pd.set_option('display.max_columns', None)

In [4]:
# import Zillow data 
zillow = gpd.read_parquet("/../../capstone/electrigrid/data/buildings/zillow.parquet").to_crs(epsg=4326)

In [5]:
# import building footprint as geopandas dataframe (may take 1-5 minutes)
building = gpd.read_parquet("../../../../../capstone/electrigrid/data/buildings/buildings_ca.parquet").to_crs(epsg=4326)

## Single and Condos Unit Adjustment

Zillow data includes condos and single family homes that have units set to greater than one. However, based on the metadata when units are greater than 1 the number of Zillow points is equal to that Zillow number. For example, if the unit is equal to 5 there would be 5 Zillow points duplicated in that point. Therefore, all units for single family homes and condos are set to one. 

In summary, the assumption is that every row that is a single family home or a condo is just one unit. 

In [6]:
## SET UNITS FOR SINGLE FAMILY HOMES AND CONDOS
# subset for single family zillow 
single_zillow = zillow[zillow['type'] == "Single"].copy()

# change all single_zillow to unit = 1
single_zillow['total_unit'] = 1

# drop the unit column
single_zillow = single_zillow.drop(['unit'], axis = 1)

# select the condo data 
zillow_condos = zillow[(zillow['code'] == "RR106") & (zillow['code'] != 'Single' )].copy()

# change all condo_zillow to unit = 1
zillow_condos['total_unit'] = 1

# drop the unit column
zillow_condos = zillow_condos.drop(['unit'], axis = 1)

assert zillow_condos.crs == single_zillow.crs

## FINAL CONCAT OF THE SINGLE HOMES WITH CONDOS
# create the complete single family homes and condo dataframe
single_condo_zillow = pd.concat([single_zillow, zillow_condos], axis = 0)

In [7]:
# view final results
single_condo_zillow.head()

,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,geometry,total_unit
10,Single,2003.0,NaN,None,None,None,NaN,None,NaN,16,06001403302,468,PGE/SCE,RR110,POINT (-122.26800 37.79429),1
16,Single,1908.0,2.0,None,None,None,36240.0,living,786.0,32,06001982000,468,PGE/SCE,RR101,POINT (-122.28185 37.80022),1
18,Single,1890.0,NaN,None,None,O,25237.0,living,736.0,36,06001982000,468,PGE/SCE,RR101,POINT (-122.28221 37.80040),1
19,Single,1896.0,2.0,None,None,None,379121.0,living,836.0,38,06001982000,468,PGE/SCE,RR101,POINT (-122.28229 37.80028),1
24,Single,NaN,NaN,None,None,None,NaN,None,NaN,51,06001982000,468,PGE/SCE,RR110,POINT (-122.28184 37.79958),1


In [8]:
# save to a parquet file 
# single_condo_zillow.to_parquet("single_condo_zillow.parquet")